# 05 – Extract Frame Features NTHU

Tujuan:
Mengekstrak vektor 512-dim untuk setiap gambar dalam dataset NTHUDDD2 menggunakan ke-4 backbone yang sudah disiapkan di Tahap 1.

Fitur ini akan disimpan ke dalam file `.pt` (PyTorch Tensor) untuk mempercepat proses windowing dan training LSTM di tahap selanjutnya.


# Import dan Setup Awal

In [1]:
# 1. IMPORT & KONFIGURASI DASAR
import os
import gc
import torch
from torch import nn
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm
import timm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device in use:", device)

# Path
BASE_DIR = r"C:\kuliah-sementara\SKRIPSI"
DATASET_NTHU_SPLIT = os.path.join(BASE_DIR, "Dataset_nthuddd2_eye_SPLIT")
MODELS_MRL_DIR     = os.path.join(BASE_DIR, "models_saved_data_MRL")
FEATURES_NTHU_DIR  = os.path.join(BASE_DIR, "features_nthuddd2")

os.makedirs(FEATURES_NTHU_DIR, exist_ok=True)  

IMG_MEAN = [0.3772, 0.3772, 0.3772]
IMG_STD  = [0.1544, 0.1544, 0.1544]
IMG_SIZE = 224

extract_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMG_MEAN, std=IMG_STD),
])


Device in use: cuda


# Definisi Ulang Arsitektur dan Fungsi Load Model

In [2]:
# 2. DEFINISI ARSITEKTUR BACKBONE DENGAN return_features=True

class SimpleCNN(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.gap = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(128, 512), nn.ReLU(), nn.Dropout(0.5), nn.Linear(512, num_classes)
        )
    def forward(self, x, return_features=False):
        x = self.gap(self.features(x))
        x = self.classifier[0](x)
        features_512 = self.classifier[3](self.classifier[2](self.classifier[1](x)))
        if return_features: return features_512
        return self.classifier[4](features_512)

class MobileNetDrowsiness(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        base = models.mobilenet_v3_small(weights=None)
        self.features, self.avgpool = base.features, base.avgpool
        self.classifier = nn.Sequential(
            nn.Dropout(p=0.3), nn.Linear(base.classifier[0].in_features, 512),
            nn.ReLU(), nn.Dropout(p=0.3), nn.Linear(512, num_classes)
        )
    def forward(self, x, return_features=False):
        x = torch.flatten(self.avgpool(self.features(x)), 1)
        features_512 = self.classifier[3](self.classifier[2](self.classifier[1](self.classifier[0](x))))
        if return_features: return features_512
        return self.classifier[4](features_512)

class VGG19Drowsiness(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        base = models.vgg19_bn(weights=None)
        self.features, self.avgpool = base.features, base.avgpool
        self.classifier = nn.Sequential(
            nn.Dropout(p=0.3), nn.Linear(base.classifier[0].in_features, 512),
            nn.ReLU(), nn.Dropout(p=0.3), nn.Linear(512, num_classes)
        )
    def forward(self, x, return_features=False):
        x = torch.flatten(self.avgpool(self.features(x)), 1)
        features_512 = self.classifier[3](self.classifier[2](self.classifier[1](self.classifier[0](x))))
        if return_features: return features_512
        return self.classifier[4](features_512)

class SwinDrowsiness(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.backbone = timm.create_model('swin_tiny_patch4_window7_224', pretrained=False, num_classes=0)
        self.classifier = nn.Sequential(
            nn.Dropout(0.3), nn.Linear(self.backbone.num_features, 512),
            nn.ReLU(), nn.Dropout(0.3), nn.Linear(512, num_classes)
        )
    def forward(self, x, return_features=False):
        x = self.backbone(x)
        features_512 = self.classifier[3](self.classifier[2](self.classifier[1](self.classifier[0](x))))
        if return_features: return features_512
        return self.classifier[4](features_512)

def load_pretrained_model(model_name):
    paths = {
        "cnn_basic": os.path.join(MODELS_MRL_DIR, "CNN_BASIC", "CNN_BASIC_BEST.pth"),
        "mobilenet": os.path.join(MODELS_MRL_DIR, "MOBILENET", "MOBILENET_BEST.pth"),
        "vgg19":     os.path.join(MODELS_MRL_DIR, "VGG19", "VGG19_BEST.pth"),
        "swin":      os.path.join(MODELS_MRL_DIR, "SWIN", "SWIN_BEST.pth")
    }
    
    if model_name == "cnn_basic": model = SimpleCNN()
    elif model_name == "mobilenet": model = MobileNetDrowsiness()
    elif model_name == "vgg19": model = VGG19Drowsiness()
    elif model_name == "swin": model = SwinDrowsiness()
    
    state = torch.load(paths[model_name], map_location="cpu")
    if "model_state_dict" in state: state = state["model_state_dict"]
    
    # Hapus prefix model. agar cocok dengan arsitektur
    new_state = {k.replace("model.", "", 1) if k.startswith("model.") else k: v for k, v in state.items()}
    
    model.load_state_dict(new_state, strict=False)
    for p in model.parameters(): p.requires_grad = False
    
    return model.to(device).eval()


# Fungsi Inti Pengekstrak Fitur

In [3]:
# 3. FUNGSI EKSTRAKSI FITUR PER DATASET SPLIT

def extract_features_to_disk(model, model_name, split_name):
    """
    Mengekstrak fitur untuk satu split (train/val/test) dan menyimpannya.
    """
    print(f"\n[{model_name.upper()}] Memulai ekstraksi fitur untuk: {split_name.upper()}")
    
    # Buat folder khusus model di dalam features_nthuddd2 (Misal: features_nthuddd2\MOBILENET)
    save_dir = os.path.join(FEATURES_NTHU_DIR, model_name.upper())
    os.makedirs(save_dir, exist_ok=True)
    
    data_dir = os.path.join(DATASET_NTHU_SPLIT, split_name)
    dataset = datasets.ImageFolder(root=data_dir, transform=extract_transform)
    
    # Batch size dibesarkan sedikit karena tidak ada backward pass (hanya inference)
    # RTX 3060 12GB sanggup menangani batch size besar.
    loader = DataLoader(dataset, batch_size=128, shuffle=False, num_workers=4, pin_memory=True)
    
    all_features = []
    all_labels = []
    all_paths = []
    
    with torch.no_grad():
        for images, labels in tqdm(loader, desc=f"Extract {split_name}"):
            images = images.to(device)
            # Forward pass minta kembalikan fitur 512-dim
            feats = model(images, return_features=True)
            
            # Pindahkan hasil ke CPU agar RAM GPU tidak penuh
            all_features.append(feats.cpu())
            all_labels.append(labels.cpu())
            
    # Dapatkan path asli gambar dari ImageFolder (penting untuk windowing LSTM nanti)
    all_paths = [path for path, _ in dataset.samples]
    
    # Gabungkan semua batch menjadi satu tensor besar
    final_features = torch.cat(all_features, dim=0)
    final_labels = torch.cat(all_labels, dim=0)
    
    # Simpan ke disk
    save_path = os.path.join(save_dir, f"{model_name}_Features_{split_name.capitalize()}_NTHUD.pt")
    
    # Kita simpan dictionary berisi tensor dan list path
    torch.save({
        "features": final_features,
        "labels": final_labels,
        "paths": all_paths
    }, save_path)
    
    print(f"Berhasil disimpan: {save_path}")
    print(f"Shape Fitur: {final_features.shape}")


# Eksekusi (Melakukan looping untuk semua model dan split)

In [4]:
# 4. EKSEKUSI EKSTRAKSI FITUR

models_to_extract = ["cnn_basic", "mobilenet", "vgg19", "swin"]
splits = ["train", "val", "test"]

for m_name in models_to_extract:
    print(f"=== Load Model {m_name.upper()} ===")
    model = load_pretrained_model(m_name)
    
    for split in splits:
        extract_features_to_disk(model, m_name, split)
        
    # Bersihkan memori GPU sebelum meload model berikutnya
    del model
    gc.collect()
    torch.cuda.empty_cache()

print("\n=== SEMUA EKSTRAKSI SELESAI ===")


=== Load Model CNN_BASIC ===

[CNN_BASIC] Memulai ekstraksi fitur untuk: TRAIN


Extract train:   0%|          | 0/319 [00:00<?, ?it/s]

Berhasil disimpan: C:\kuliah-sementara\SKRIPSI\features_nthuddd2\CNN_BASIC\cnn_basic_Features_Train_NTHUD.pt
Shape Fitur: torch.Size([40811, 512])

[CNN_BASIC] Memulai ekstraksi fitur untuk: VAL


Extract val:   0%|          | 0/144 [00:00<?, ?it/s]

Berhasil disimpan: C:\kuliah-sementara\SKRIPSI\features_nthuddd2\CNN_BASIC\cnn_basic_Features_Val_NTHUD.pt
Shape Fitur: torch.Size([18419, 512])

[CNN_BASIC] Memulai ekstraksi fitur untuk: TEST


...

Berhasil disimpan: C:\kuliah-sementara\SKRIPSI\features_nthuddd2\SWIN\swin_Features_Train_NTHUD.pt
Shape Fitur: torch.Size([40811, 512])

[SWIN] Memulai ekstraksi fitur untuk: VAL


Extract val:   0%|          | 0/144 [00:00<?, ?it/s]

Berhasil disimpan: C:\kuliah-sementara\SKRIPSI\features_nthuddd2\SWIN\swin_Features_Val_NTHUD.pt
Shape Fitur: torch.Size([18419, 512])

[SWIN] Memulai ekstraksi fitur untuk: TEST


Extract test:   0%|          | 0/52 [00:00<?, ?it/s]

Berhasil disimpan: C:\kuliah-sementara\SKRIPSI\features_nthuddd2\SWIN\swin_Features_Test_NTHUD.pt
Shape Fitur: torch.Size([6633, 512])

=== SEMUA EKSTRAKSI SELESAI ===
